# YOLO SAM31 P2 Live Training

Run this notebook on the GPU cluster after cloning/pulling the repo. It trains YOLO segmentation on `training_data/dataset/yolo_seg_dataset` and writes all run outputs to `outputs/yolo_cluster_live/`.

In [ ]:
from pathlib import Path
import csv
import json
import os
import shlex
import shutil
import subprocess
import sys
import time
from datetime import datetime

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, clear_output, display

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'training_data/dataset/yolo_seg_dataset').exists():
            return candidate
    raise FileNotFoundError('Could not find training_data/dataset/yolo_seg_dataset from current path')

REPO_ROOT = find_repo_root(Path.cwd())
YOLO_DATASET = REPO_ROOT / 'training_data/dataset/yolo_seg_dataset'
OUTPUT_ROOT = REPO_ROOT / 'outputs/yolo_cluster_live'
REFERENCE_MODEL_DIR = REPO_ROOT / 'training_data/reference_models'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
REFERENCE_MODEL_DIR.mkdir(parents=True, exist_ok=True)

EPOCHS = 120
IMGSZ = 512
BATCH = 8
WORKERS = 4
PATIENCE = 35
MODEL = REPO_ROOT / 'yolov8s-seg.pt'
if not MODEL.exists():
    MODEL = 'yolov8s-seg.pt'

DATASET_IMAGE_FILES = sorted(p for p in (YOLO_DATASET / 'images').glob('*/*.png') if not p.name.startswith('._'))
DATASET_LABEL_FILES = sorted(p for p in (YOLO_DATASET / 'labels').glob('*/*.txt') if not p.name.startswith('._'))
DATASET_TILE_COUNT = len(DATASET_IMAGE_FILES)

def has_nvidia_gpu() -> bool:
    try:
        result = subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=5)
        return result.returncode == 0
    except Exception:
        return False

DEVICE = '0' if has_nvidia_gpu() else 'cpu'
RUN_NAME = f'yolo_sam31_p2_{DATASET_TILE_COUNT}tiles_' + datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR = OUTPUT_ROOT / RUN_NAME
LIVE_LOG = OUTPUT_ROOT / f'{RUN_NAME}_live_training.log'
RUNTIME_DATA_YAML = OUTPUT_ROOT / 'data_cluster_runtime.yaml'

print('REPO_ROOT:', REPO_ROOT)
print('YOLO_DATASET:', YOLO_DATASET)
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('MODEL:', MODEL)
print('DEVICE:', DEVICE)
print('DATASET_TILE_COUNT:', DATASET_TILE_COUNT)
print('RUN_NAME:', RUN_NAME)

In [ ]:
try:
    import ultralytics
    print('ultralytics:', ultralytics.__version__)
except Exception as exc:
    raise RuntimeError('Ultralytics is not installed in this environment. Install it once with: python -m pip install ultralytics') from exc

image_files = DATASET_IMAGE_FILES
label_files = DATASET_LABEL_FILES
assert image_files, f'No YOLO images found under {YOLO_DATASET / "images"}'
assert len(image_files) == len(label_files), f'Image/label count mismatch: {len(image_files)} images vs {len(label_files)} labels'
image_stems = {p.stem for p in image_files}
label_stems = {p.stem for p in label_files}
missing_labels = sorted(image_stems - label_stems)
missing_images = sorted(label_stems - image_stems)
assert not missing_labels and not missing_images, {
    'missing_labels': missing_labels,
    'missing_images': missing_images,
}
print(f'YOLO dataset ready: {len(image_files)} images, {len(label_files)} labels')

runtime_yaml_text = f'''path: {YOLO_DATASET}
train: images/train
val: images/val

names:
  0: nucleus
  1: clear_cell_boundary
  2: compact_cell_boundary
  3: stroma
'''
RUNTIME_DATA_YAML.write_text(runtime_yaml_text, encoding='utf-8')

print(RUNTIME_DATA_YAML)
print(RUNTIME_DATA_YAML.read_text())
display(pd.read_csv(YOLO_DATASET / 'label_counts.csv'))

In [ ]:
def nvidia_snapshot() -> str:
    try:
        result = subprocess.run([
            'nvidia-smi',
            '--query-gpu=name,memory.used,memory.total,utilization.gpu,temperature.gpu',
            '--format=csv,noheader,nounits',
        ], capture_output=True, text=True, timeout=5)
        return result.stdout.strip() if result.returncode == 0 else result.stderr.strip()
    except Exception as exc:
        return f'nvidia-smi unavailable: {exc}'

def tail_text(path: Path, lines: int = 18) -> str:
    if not path.exists():
        return '(log not created yet)'
    text = path.read_text(errors='replace')
    return '\n'.join(text.splitlines()[-lines:])

def load_results(run_dir: Path):
    path = run_dir / 'results.csv'
    if not path.exists():
        return None
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    return df

def plot_live_results(df: pd.DataFrame, out_path: Path):
    if df is None or df.empty:
        return
    x = df['epoch'] if 'epoch' in df.columns else range(len(df))
    loss_cols = [c for c in df.columns if 'loss' in c.lower()]
    metric_cols = [c for c in df.columns if c.startswith('metrics/') or 'mAP' in c or 'precision' in c or 'recall' in c]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for col in loss_cols:
        axes[0].plot(x, df[col], label=col.replace('train/', '').replace('val/', 'val '))
    axes[0].set_title('YOLO losses')
    axes[0].set_xlabel('epoch')
    axes[0].grid(alpha=0.25)
    axes[0].legend(fontsize=8)
    for col in metric_cols:
        axes[1].plot(x, df[col], label=col.replace('metrics/', ''))
    axes[1].set_title('Validation metrics')
    axes[1].set_xlabel('epoch')
    axes[1].grid(alpha=0.25)
    axes[1].legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(out_path, dpi=160)
    plt.show()

def live_dashboard(run_dir: Path, log_path: Path):
    clear_output(wait=True)
    print('Run:', run_dir)
    print('Time:', datetime.now().isoformat(timespec='seconds'))
    print('GPU:', nvidia_snapshot())
    df = load_results(run_dir)
    if df is not None and not df.empty:
        display(df.tail(5))
        plot_live_results(df, run_dir / 'live_metrics.png')
    else:
        print('Waiting for results.csv...')
    print('--- log tail ---')
    print(tail_text(log_path))

In [ ]:
cmd = [
    sys.executable, '-m', 'ultralytics', 'segment', 'train',
    f'model={MODEL}',
    f'data={RUNTIME_DATA_YAML}',
    f'epochs={EPOCHS}',
    f'imgsz={IMGSZ}',
    f'batch={BATCH}',
    f'device={DEVICE}',
    f'workers={WORKERS}',
    f'patience={PATIENCE}',
    'optimizer=AdamW',
    'lr0=0.002',
    'mosaic=1.0',
    'close_mosaic=20',
    'copy_paste=0.1',
    'degrees=10',
    'translate=0.08',
    'scale=0.35',
    'fliplr=0.5',
    f'project={OUTPUT_ROOT}',
    f'name={RUN_NAME}',
]

print('Command:')
print(' '.join(shlex.quote(str(x)) for x in cmd))

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env.setdefault('WANDB_MODE', 'offline')

with LIVE_LOG.open('w', encoding='utf-8') as log_handle:
    proc = subprocess.Popen(cmd, cwd=REPO_ROOT, stdout=log_handle, stderr=subprocess.STDOUT, text=True, env=env)
    while proc.poll() is None:
        live_dashboard(RUN_DIR, LIVE_LOG)
        time.sleep(15)
    return_code = proc.wait()

live_dashboard(RUN_DIR, LIVE_LOG)
if return_code != 0:
    raise RuntimeError(f'YOLO training failed with return code {return_code}. See {LIVE_LOG}')
print('Training complete')

In [ ]:
best_pt = RUN_DIR / 'weights/best.pt'
last_pt = RUN_DIR / 'weights/last.pt'
deployed_pt = REFERENCE_MODEL_DIR / f'yolo_sam31_p2_{DATASET_TILE_COUNT}tiles_best.pt'
if best_pt.exists():
    shutil.copy2(best_pt, deployed_pt)

summary = {
    'timestamp': datetime.now().isoformat(timespec='seconds'),
    'repo_root': str(REPO_ROOT),
    'dataset': str(YOLO_DATASET),
    'runtime_data_yaml': str(RUNTIME_DATA_YAML),
    'run_dir': str(RUN_DIR),
    'live_log': str(LIVE_LOG),
    'best_pt': str(best_pt),
    'last_pt': str(last_pt),
    'deployed_pt': str(deployed_pt) if best_pt.exists() else '',
    'epochs': EPOCHS,
    'imgsz': IMGSZ,
    'batch': BATCH,
    'device': DEVICE,
    'dataset_tile_count': DATASET_TILE_COUNT,
}
(RUN_DIR / 'yolo_live_training_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
display(Markdown('## Final files'))
for key, value in summary.items():
    print(f'{key}: {value}')
print('Available plots:', [p.name for p in RUN_DIR.glob('*.png')])

In [ ]:
# Optional qualitative check after training: generate predictions on validation tiles.
from ultralytics import YOLO

assert best_pt.exists(), best_pt
model = YOLO(str(best_pt))
pred = model.predict(
    source=str(YOLO_DATASET / 'images/val'),
    imgsz=IMGSZ,
    conf=0.25,
    save=True,
    project=str(OUTPUT_ROOT),
    name=RUN_NAME + '_predict_val',
)
print('Prediction output:', OUTPUT_ROOT / (RUN_NAME + '_predict_val'))